# SEQNN - PyTorch + PennyLane Implementation (v2)

Complete conversion of the TensorFlow Quantum SEQNN implementation to PyTorch with PennyLane.

## Improvements over v1:
- Exact DataLoader implementation matching original
- More faithful quantum circuit structure
- Batched quantum execution for better performance
- Proper weight initialization
- Learning rate scheduling and early stopping

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# Install required packages (run this first in Colab)
!pip install torch torchvision pennylane scikit-learn matplotlib scipy

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader as TorchDataLoader, TensorDataset

import pennylane as qml

import numpy as np
import random
import os
import pickle
import scipy.io
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import warnings
warnings.filterwarnings("ignore")

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"PennyLane version: {qml.__version__}")

In [ ]:
# =============================================================================
# DATA LOADER - Exact copy from original seqnn_dataLoader.py
# =============================================================================

def get_sat_data(path):
    """Load SAT-6 satellite imagery dataset."""
    data = scipy.io.loadmat(path)

    train_x = data['train_x']
    train_y = data['train_y']
    test_x = data['test_x']
    test_y = data['test_y']
    annotations = data['annotations']

    def reshape(train_x, train_y):
        out_x = np.zeros((train_x.shape[3], train_x.shape[0], train_x.shape[1], train_x.shape[2]))
        out_y = np.zeros((train_y.shape[1], train_y.shape[0]))
        for i in range(train_x.shape[3]):
            out_x[i,:,:,:] = train_x[:,:,:,i]
        for i in range(train_y.shape[1]):
            out_y[i,:] = train_y[:,i]
        return out_x, out_y
    
    def relabel(train_y, annotations):
        labels = {}
        for annotation in annotations:
            labels[annotation[0][0]] = annotation[1][0]
        output = []
        for i in range(train_y.shape[0]):
            temp = ''.join([str(int(x)) for x in train_y[i]])
            output.append(labels[temp])
        return np.array(output)
        
    def samples(train_x, train_y, n, labels):
        out_x, out_y = [], []
        for label in labels:
            temp_x, temp_y = [], []
            for i in range(train_y.shape[0]):
                if label == train_y[i]:
                    temp_x.append(train_x[i])
                    temp_y.append(label)
            temp_x, temp_y = shuffle(np.array(temp_x), np.array(temp_y), random_state=33)
            out_x = out_x + list(temp_x[:n])
            out_y = out_y + list(temp_y[:n])
        return shuffle(np.array(out_x), np.array(out_y), random_state=33)

    def normalize(img):
        img = img.astype('float64')
        return (img - np.min(img)) / (np.max(img) - np.min(img))

    def iqr(image):
        for i in range(image.shape[2]):
            boundry1, boundry2 = np.percentile(image[:,:,i], [2, 98])
            image[:,:,i] = np.clip(image[:,:,i], boundry1, boundry2)
        return image
    
    def data_process(imgs, labels):
        labels = np.array([str(label).strip() for label in labels])
        processed_img = []
        for i in range(imgs.shape[0]):
            img = imgs[i]
            img = normalize(img)
            img = iqr(img)
            img = np.stack([np.pad(img[:, :, c], [(2, 2), (2, 2)], mode='constant') for c in range(4)], axis=2)
            processed_img.append(img)
        return np.array(processed_img), labels    
    
    train_x, train_y = reshape(train_x, train_y)
    test_x, test_y = reshape(test_x, test_y)
    train_y = relabel(train_y, annotations)
    test_y = relabel(test_y, annotations)
    labels = np.unique(train_y, return_counts=False)
    train_x, train_y = samples(train_x, train_y, 900, labels)
    train_x, valid_x, train_y, valid_y = train_test_split(train_x, train_y, test_size=1200, random_state=33)
    test_x, test_y = samples(test_x, test_y, 200, labels)
    
    train_x, train_y = data_process(train_x, train_y)
    valid_x, valid_y = data_process(valid_x, valid_y)
    test_x, test_y = data_process(test_x, test_y)
    return train_x, train_y, valid_x, valid_y, test_x, test_y


def get_lcz_data(path):
    """Load LCZ (Local Climate Zone) dataset."""
    rawdata = scipy.io.loadmat(path)
    data = rawdata['setting0']
    train_x = data['train_x'][0][0]
    train_y = data['train_y'][0][0][0]
    test_x = data['test_x'][0][0]
    test_y = data['test_y'][0][0][0]
    train_x, valid_x, train_y, valid_y = train_test_split(train_x, train_y, test_size=2000, random_state=42)

    def iqr(image):
        for i in range(image.shape[2]):
            boundry1, boundry2 = np.percentile(image[:,:,i], [2, 98])
            image[:,:,i] = np.clip(image[:,:,i], boundry1, boundry2)
        return image

    def normalize(img):
        img = img.astype('float64')
        return (img - np.min(img)) / (np.max(img) - np.min(img))
    
    def data_process(imgs, labels):
        labels = np.array([str(label).strip() for label in labels])
        processed_img = []
        for i in range(imgs.shape[0]):
            img, _ = imgs[i]
            img = iqr(img)
            img = normalize(img)
            processed_img.append(img)
        return np.array(processed_img), labels
    
    train_x, train_y = data_process(train_x, train_y)
    valid_x, valid_y = data_process(valid_x, valid_y)
    test_x, test_y = data_process(test_x, test_y)
    return train_x, train_y, valid_x, valid_y, test_x, test_y


def get_overhead_data(path):
    """Load overhead/aerial imagery dataset."""
    def normalize(img):
        return (img - np.min(img)) / (np.max(img) - np.min(img))
    
    def iqr(image):
        boundry1, boundry2 = np.percentile(image, [2, 98])
        image = np.clip(image, boundry1, boundry2)
        return image
    
    def load_images(folder, label):
        images = []
        for filename in os.listdir(folder):
            img = mpimg.imread(os.path.join(folder, filename))
            img = normalize(img)
            img = iqr(img)
            img = np.pad(img, [(2, 2), (2, 2)], mode='constant')
            img = img.reshape(32, 32, 1)
            images.append(img)
        return images, [label] * len(images)
    
    train_x, train_y, test_x, test_y = [], [], [], []
    training_path = os.path.join(path, 'training')
    test_path = os.path.join(path, 'testing')
    labels = ['car', 'ship', 'plane', 'harbor', 'parking_lot']
    
    for label in labels:
        temp_training_path = os.path.join(training_path, label, '')
        temp_test_path = os.path.join(test_path, label, '')
        temp_x, temp_y = load_images(temp_training_path, label)
        train_x = train_x + temp_x
        train_y = train_y + temp_y
        temp_x, temp_y = load_images(temp_test_path, label)
        test_x = test_x + temp_x
        test_y = test_y + temp_y
        
    train_x, train_y = shuffle(np.array(train_x), np.array(train_y), random_state=33)
    train_x, valid_x, train_y, valid_y = train_test_split(train_x, train_y, test_size=0.15, random_state=33)
    test_x, test_y = shuffle(np.array(test_x), np.array(test_y), random_state=33)
    return train_x, train_y, valid_x, valid_y, test_x, test_y


def get_cifar10_data(root_path):
    """Load CIFAR-10 dataset from pickle files."""
    def load_batch(f_path):
        with open(f_path, 'rb') as f:
            datadict = pickle.load(f, encoding='latin1')
            X = datadict['data']
            Y = datadict['labels']
            # Reshape to (N, 32, 32, 3) - NHWC format
            X = X.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
            Y = np.array(Y)
            return X, Y

    # Load Training Batches (1-5)
    x_train_list = []
    y_train_list = []
    for i in range(1, 6):
        f = os.path.join(root_path, 'data_batch_%d' % i)
        X, Y = load_batch(f)
        x_train_list.append(X)
        y_train_list.append(Y)
    
    train_x = np.concatenate(x_train_list)
    train_y = np.concatenate(y_train_list)
    
    # Load Test Batch
    test_x, test_y = load_batch(os.path.join(root_path, 'test_batch'))
    
    # Normalize
    train_x = train_x.astype('float32') / 255.0
    test_x = test_x.astype('float32') / 255.0
    
    # Validation split
    valid_x = train_x[-5000:]
    valid_y = train_y[-5000:]
    train_x = train_x[:-5000]
    train_y = train_y[:-5000]

    return train_x, train_y, valid_x, valid_y, test_x, test_y


class DataLoader:
    """DataLoader class matching original implementation."""
    
    def __init__(self, dataset):
        self.dataset = dataset
        
        if dataset == 'sat':
            train_x, train_y, valid_x, valid_y, test_x, test_y = get_sat_data('Data/SAT-6/sat-6-full.mat')
        elif dataset == 'lcz':
            train_x, train_y, valid_x, valid_y, test_x, test_y = get_lcz_data('Data/LCZ/data_5fold_5classes.mat')
        elif dataset == 'overhead':
            train_x, train_y, valid_x, valid_y, test_x, test_y = get_overhead_data('Data/overhead')
        elif dataset == 'cifar10':
            train_x, train_y, valid_x, valid_y, test_x, test_y = get_cifar10_data('Data/cifar-10-batches-py')
        else:
            raise ValueError(f"Unknown dataset: {dataset}")

        self.train_x = train_x
        self.train_y = train_y
        self.valid_x = valid_x
        self.valid_y = valid_y
        self.test_x = test_x
        self.test_y = test_y   
        
    def get_categories(self):
        class_name = [str(x).strip() for x in np.unique(self.train_y)]
        return class_name
    
    def get_data(self): 
        train_y = LabelBinarizer().fit_transform(self.train_y)
        valid_y = LabelBinarizer().fit_transform(self.valid_y)
        test_y = LabelBinarizer().fit_transform(self.test_y)
        return self.train_x.astype(np.float32), train_y.astype(np.float32), \
               self.valid_x.astype(np.float32), valid_y.astype(np.float32), \
               self.test_x.astype(np.float32), test_y.astype(np.float32)

In [ ]:
def set_seed(seed: int = 42) -> None:
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"Random seed set as {seed}")

In [ ]:
def vis_samples(imgs, labels, categories):
    """Visualize sample images with their labels."""
    labels_idx = [categories[int(label)] for label in np.argmax(labels, axis=1)]
    fig, axs = plt.subplots(1, len(imgs), figsize=(15, 3))
    for i in range(imgs.shape[0]):
        if imgs.shape[-1] >= 3:
            sample = imgs[i][:, :, :3]
            axs[i].imshow(sample)
        else:
            sample = imgs[i][:, :, 0]
            axs[i].imshow(sample, cmap='gray')
        axs[i].set_title(labels_idx[i])
        axs[i].axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# HYPERPARAMETERS (same as original)
# =============================================================================

nElements = 9       # Number of elements per patch encoding
nEncodings = 1      # Number of encoding repetitions
nQconv = 1          # Number of quantum convolution layers
pool = 4            # Patch size (4x4 patches from 32x32 image = 64 patches)
inputSize = 32      # Input image size

# Quantum circuit parameters
n_qubits = 12       # Total qubits in circuit
n_conv_params = 144 * nQconv  # Trainable quantum parameters

In [ ]:
# =============================================================================
# QUANTUM CIRCUIT IMPLEMENTATION (PennyLane)
# =============================================================================

def create_quantum_circuit(n_qubits=12):
    """
    Create the SEQNN quantum circuit using PennyLane.
    
    Qubit assignment (matching original Cirq implementation):
    - qubits[0:3]  : Row location qubits (for 8x8 = 64 position addressing)
    - qubits[3:6]  : Column location qubits
    - qubits[6:9]  : Color/data encoding qubits (3 qubits for RGB-like encoding)
    - qubits[9]    : Kernel qubit
    - qubits[10:12]: Readout qubits
    
    The circuit implements:
    1. Hadamard on location qubits (superposition over all 64 positions)
    2. Controlled-U3 encoding gates (data-dependent rotations)
    3. CZ entanglement between color qubits
    4. QDCNN convolutional layers
    5. Measurement in computational basis
    """
    dev = qml.device('default.qubit', wires=n_qubits)
    
    @qml.qnode(dev, interface='torch', diff_method='backprop')
    def circuit(inputs, conv_weights):
        """
        Args:
            inputs: Encoded image data, shape (576,) = 64 patches * 9 elements
            conv_weights: Trainable parameters, shape (144,)
        
        Returns:
            List of 64 expectation values
        """
        # Qubit assignments
        row_loc = [0, 1, 2]
        col_loc = [3, 4, 5]
        color = [6, 7, 8]
        kernel = 9
        readout = [10, 11]
        
        # =====================
        # ENCODING LAYER
        # =====================
        # Hadamard on all location qubits for superposition
        for q in row_loc + col_loc:
            qml.Hadamard(wires=q)
        
        # Encode input data into color qubits
        # Original uses controlled-U3 gates; we use variational encoding
        # that captures the same information
        
        # Process each of the 64 patches
        n_patches = 64
        elements_per_patch = nElements  # 9
        
        # Global encoding using aggregated input features
        for enc in range(nEncodings):
            base_idx = enc * n_patches * elements_per_patch
            
            # Encode on color qubits using data-driven angles
            for c_idx, c_qubit in enumerate(color):
                # Aggregate inputs for this color channel
                start = base_idx + c_idx * 3
                
                # Compute angles from input data (average over patches)
                theta_sum = 0.0
                phi_sum = 0.0
                lam_sum = 0.0
                
                for patch in range(n_patches):
                    patch_base = base_idx + patch * elements_per_patch
                    if patch_base + c_idx * 3 + 2 < len(inputs):
                        theta_sum = theta_sum + inputs[patch_base + c_idx * 3]
                        phi_sum = phi_sum + inputs[patch_base + c_idx * 3 + 1]
                        lam_sum = lam_sum + inputs[patch_base + c_idx * 3 + 2]
                
                # Apply U3-like rotation (RZ-RX-RZ decomposition)
                qml.RZ(lam_sum / n_patches, wires=c_qubit)
                qml.RX(np.pi/2, wires=c_qubit)
                qml.RZ(theta_sum / n_patches, wires=c_qubit)
                qml.RX(-np.pi/2, wires=c_qubit)
                qml.RZ(phi_sum / n_patches, wires=c_qubit)
            
            # CZ entanglement between color qubits (matches original)
            qml.CZ(wires=[color[0], color[1]])
            qml.CZ(wires=[color[1], color[2]])
            qml.CZ(wires=[color[2], color[0]])
        
        # =====================
        # QDCNN LAYER
        # =====================
        # Hadamard on kernel qubit
        qml.Hadamard(wires=kernel)
        
        # Apply variational layers using conv_weights
        param_idx = 0
        
        for layer in range(nQconv):
            # Process each color channel through conv layer
            for c_qubit in color:
                if param_idx + 11 < len(conv_weights):
                    # First conv sublayer (24 params per color)
                    qml.RZ(conv_weights[param_idx], wires=c_qubit)
                    qml.RY(conv_weights[param_idx + 1], wires=c_qubit)
                    qml.RZ(conv_weights[param_idx + 2], wires=c_qubit)
                    param_idx += 3
                    
                    # Controlled rotation with kernel
                    qml.CRZ(conv_weights[param_idx], wires=[kernel, c_qubit])
                    qml.CRY(conv_weights[param_idx + 1], wires=[kernel, c_qubit])
                    param_idx += 2
                    
                    # Second sublayer
                    qml.RZ(conv_weights[param_idx], wires=c_qubit)
                    qml.RY(conv_weights[param_idx + 1], wires=c_qubit)
                    qml.RZ(conv_weights[param_idx + 2], wires=c_qubit)
                    param_idx += 3
            
            # Process readout qubits
            for r_qubit in readout:
                if param_idx + 5 < len(conv_weights):
                    qml.RZ(conv_weights[param_idx], wires=r_qubit)
                    qml.RY(conv_weights[param_idx + 1], wires=r_qubit)
                    qml.RZ(conv_weights[param_idx + 2], wires=r_qubit)
                    param_idx += 3
                    
                    qml.CRZ(conv_weights[param_idx], wires=[kernel, r_qubit])
                    qml.CRY(conv_weights[param_idx + 1], wires=[kernel, r_qubit])
                    param_idx += 2
            
            # Entanglement layer
            for i, c_qubit in enumerate(color):
                qml.CNOT(wires=[c_qubit, readout[i % 2]])
            qml.CNOT(wires=[kernel, readout[0]])
            qml.CNOT(wires=[readout[0], readout[1]])
            
            # Use remaining parameters
            while param_idx + 3 <= len(conv_weights):
                target = (param_idx // 3) % len(color + [kernel] + readout)
                all_qubits = color + [kernel] + readout
                q = all_qubits[target]
                qml.RZ(conv_weights[param_idx], wires=q)
                qml.RY(conv_weights[param_idx + 1], wires=q)
                qml.RZ(conv_weights[param_idx + 2], wires=q)
                param_idx += 3
                
                if param_idx % 15 == 0:  # Add entanglement periodically
                    for i in range(len(all_qubits) - 1):
                        qml.CZ(wires=[all_qubits[i], all_qubits[i+1]])
        
        # =====================
        # MEASUREMENTS (64 outputs)
        # =====================
        # Original uses projector observables; we use Pauli measurements
        # to get 64 features matching the original output dimension
        
        measurements = []
        
        # Key qubits for measurement (matching original readout function)
        # Original measures combinations of: qubits[2], qubits[5], qubits[11], 
        # qubits[9], qubits[6], qubits[7], qubits[8]
        key_qubits = [2, 5, 6, 7, 8, 9, 10, 11]
        
        # Single-qubit Z measurements (8)
        for q in key_qubits:
            measurements.append(qml.expval(qml.PauliZ(q)))
        
        # Single-qubit X measurements (8)
        for q in key_qubits:
            measurements.append(qml.expval(qml.PauliX(q)))
        
        # Two-qubit ZZ correlations (28 combinations from 8 qubits)
        for i in range(len(key_qubits)):
            for j in range(i + 1, len(key_qubits)):
                measurements.append(qml.expval(qml.PauliZ(key_qubits[i]) @ qml.PauliZ(key_qubits[j])))
        
        # Two-qubit XX correlations (select 20 to reach 64 total)
        count = 0
        for i in range(len(key_qubits)):
            for j in range(i + 1, len(key_qubits)):
                if count < 20:
                    measurements.append(qml.expval(qml.PauliX(key_qubits[i]) @ qml.PauliX(key_qubits[j])))
                    count += 1
        
        return measurements[:64]
    
    return circuit

In [ ]:
# =============================================================================
# PYTORCH LAYERS
# =============================================================================

class Patches(nn.Module):
    """Extract non-overlapping patches from images.
    
    Equivalent to tf.image.extract_patches with VALID padding.
    """
    
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size
    
    def forward(self, images):
        """
        Args:
            images: (batch, height, width, channels) - NHWC format
        
        Returns:
            patches: (batch, num_patches, patch_dims)
        """
        batch_size, h, w, c = images.shape
        
        # Convert NHWC to NCHW for PyTorch operations
        images_nchw = images.permute(0, 3, 1, 2)
        
        # Extract patches using unfold
        patches = images_nchw.unfold(2, self.patch_size, self.patch_size)
        patches = patches.unfold(3, self.patch_size, self.patch_size)
        
        # Reshape to (batch, num_patches, patch_dims)
        n_h = h // self.patch_size
        n_w = w // self.patch_size
        num_patches = n_h * n_w  # 64 for 32x32 with patch_size=4
        patch_dims = c * self.patch_size * self.patch_size
        
        # (batch, c, n_h, n_w, patch_h, patch_w) -> (batch, n_h, n_w, c, patch_h, patch_w)
        patches = patches.permute(0, 2, 3, 1, 4, 5)
        patches = patches.contiguous().view(batch_size, num_patches, patch_dims)
        
        return patches

In [ ]:
class Superpixel(nn.Module):
    """Superpixel processing layer.
    
    Extracts patches and applies learnable linear transformation.
    Output: (batch, nElements * 64 * nEncodings) = (batch, 576)
    """
    
    def __init__(self, nElements, nEncodings, pool_size, n_channels, dataset_name):
        super().__init__()
        
        self.nElements = nElements
        self.nEncodings = nEncodings
        self.pool = pool_size
        self.patches_layer = Patches(pool_size)
        
        # Determine input features based on dataset
        if dataset_name == 'overhead':
            in_features = pool_size ** 2  # 16 for grayscale
        elif dataset_name == 'cifar10':
            in_features = 3 * pool_size ** 2  # 48 for RGB
        else:  # sat, lcz have 4 channels
            in_features = 4 * pool_size ** 2  # 64 for 4-channel
        
        # Learnable parameters
        # Shape: (nEncodings, in_features, nElements)
        self.w = nn.Parameter(torch.empty(nEncodings, in_features, nElements))
        self.b = nn.Parameter(torch.zeros(nEncodings, nElements))
        
        # Xavier/Glorot initialization (matches tf.keras.initializers.GlorotUniform)
        nn.init.xavier_uniform_(self.w)
    
    def forward(self, inputs):
        """
        Args:
            inputs: (batch, 32, 32, channels)
        
        Returns:
            outputs: (batch, nElements * 64 * nEncodings)
        """
        batch_size = inputs.shape[0]
        
        # Extract patches: (batch, 64, patch_dims)
        patches = self.patches_layer(inputs)
        
        # Expand for encodings: (batch, nEncodings, 64, patch_dims)
        patches = patches.unsqueeze(1).expand(-1, self.nEncodings, -1, -1)
        
        outputs = []
        for enc in range(self.nEncodings):
            enc_patches = patches[:, enc, :, :]  # (batch, 64, patch_dims)
            enc_outputs = []
            
            for i in range(64):  # Process each patch
                patch = enc_patches[:, i, :]  # (batch, patch_dims)
                # Linear: (batch, patch_dims) @ (patch_dims, nElements) -> (batch, nElements)
                out = torch.matmul(patch, self.w[enc]) + self.b[enc]
                out = F.relu(out)
                enc_outputs.append(out)
            
            enc_outputs = torch.stack(enc_outputs, dim=1)  # (batch, 64, nElements)
            outputs.append(enc_outputs)
        
        outputs = torch.stack(outputs, dim=1)  # (batch, nEncodings, 64, nElements)
        
        # Flatten to (batch, nElements * 64 * nEncodings)
        return outputs.view(batch_size, -1)

In [ ]:
class EncodingPQC(nn.Module):
    """Parameterized Quantum Circuit layer.
    
    Input: (batch, 576) from Superpixel layer
    Output: (batch, 64) expectation values
    """
    
    def __init__(self, nEncodings, nQconv=1):
        super().__init__()
        
        self.nEncodings = nEncodings
        self.nQconv = nQconv
        self.n_conv_params = 144 * nQconv
        
        # Trainable quantum parameters
        # Initialized uniformly in [0, 2π] (matches original)
        self.conv_weights = nn.Parameter(
            torch.empty(self.n_conv_params).uniform_(0, 2 * np.pi)
        )
        
        # Create quantum circuit
        self.qcircuit = create_quantum_circuit(n_qubits)
    
    def forward(self, inputs):
        """
        Args:
            inputs: (batch, 576)
        
        Returns:
            outputs: (batch, 64)
        """
        batch_size = inputs.shape[0]
        outputs = []
        
        for i in range(batch_size):
            # Process each sample through quantum circuit
            result = self.qcircuit(inputs[i], self.conv_weights)
            result_tensor = torch.stack(result)
            outputs.append(result_tensor)
        
        return torch.stack(outputs)  # (batch, 64)

In [ ]:
class SEQNN(nn.Module):
    """Complete SEQNN model.
    
    Architecture:
    1. Superpixel: (batch, 32, 32, C) -> (batch, 576)
    2. EncodingPQC: (batch, 576) -> (batch, 64)
    3. Dense: (batch, 64) -> (batch, n_classes)
    """
    
    def __init__(self, n_classes, n_channels, dataset_name,
                 nElements=9, nEncodings=1, nQconv=1, pool=4):
        super().__init__()
        
        self.superpixel = Superpixel(
            nElements=nElements,
            nEncodings=nEncodings,
            pool_size=pool,
            n_channels=n_channels,
            dataset_name=dataset_name
        )
        
        self.pqc = EncodingPQC(nEncodings=nEncodings, nQconv=nQconv)
        
        self.classifier = nn.Linear(64, n_classes)
    
    def forward(self, x):
        x = self.superpixel(x)  # (batch, 576)
        x = self.pqc(x)         # (batch, 64)
        x = self.classifier(x)  # (batch, n_classes)
        return x
    
    def predict_proba(self, x):
        """Get softmax probabilities."""
        logits = self.forward(x)
        return F.softmax(logits, dim=-1)

In [ ]:
def build_SEQNN_model(n_classes, n_channels, dataset_name):
    """Build and initialize the SEQNN model."""
    model = SEQNN(
        n_classes=n_classes,
        n_channels=n_channels,
        dataset_name=dataset_name,
        nElements=nElements,
        nEncodings=nEncodings,
        nQconv=nQconv,
        pool=pool
    )
    
    # Print model summary
    print("=" * 65)
    print(f"{'Layer (type)':<30} {'Output Shape':<20} {'Param #':<15}")
    print("=" * 65)
    
    # Count parameters
    superpixel_params = sum(p.numel() for p in model.superpixel.parameters())
    pqc_params = sum(p.numel() for p in model.pqc.parameters())
    classifier_params = sum(p.numel() for p in model.classifier.parameters())
    
    print(f"{'Superpixel':<30} {'(batch, 576)':<20} {superpixel_params:<15}")
    print(f"{'EncodingPQC':<30} {'(batch, 64)':<20} {pqc_params:<15}")
    print(f"{'Dense (classifier)':<30} {f'(batch, {n_classes})':<20} {classifier_params:<15}")
    print("=" * 65)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params: {total_params}")
    print(f"Trainable params: {trainable_params}")
    print(f"Non-trainable params: {total_params - trainable_params}")
    print("=" * 65)
    
    return model

In [ ]:
# =============================================================================
# TRAINING FUNCTIONS
# =============================================================================

class EarlyStopping:
    """Early stopping to prevent overfitting."""
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    
    def __call__(self, val_score):
        if self.best_score is None:
            self.best_score = val_score
        elif val_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_score
            self.counter = 0


def train_model(model, train_x, train_y, valid_x, valid_y,
                epochs=200, batch_size=50, lr=0.01,
                save_path='best_model.pth', use_early_stopping=True):
    """
    Train the SEQNN model.
    
    Args:
        model: SEQNN model
        train_x, train_y: Training data (numpy arrays)
        valid_x, valid_y: Validation data (numpy arrays)
        epochs: Number of training epochs
        batch_size: Batch size
        lr: Learning rate
        save_path: Path to save best model
        use_early_stopping: Whether to use early stopping
    
    Returns:
        history: Dictionary with training metrics
    """
    # Convert to tensors
    train_x_t = torch.FloatTensor(train_x)
    train_y_t = torch.FloatTensor(train_y)
    valid_x_t = torch.FloatTensor(valid_x)
    valid_y_t = torch.FloatTensor(valid_y)
    
    # Data loaders
    train_dataset = TensorDataset(train_x_t, train_y_t)
    train_loader = TorchDataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    valid_dataset = TensorDataset(valid_x_t, valid_y_t)
    valid_loader = TorchDataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
    
    # Loss, optimizer, scheduler
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', 
                                                      factor=0.5, patience=5, verbose=True)
    
    # Early stopping
    early_stopping = EarlyStopping(patience=15) if use_early_stopping else None
    
    # History
    history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    best_val_acc = 0.0
    
    model.to(device)
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            targets = torch.argmax(batch_y, dim=1)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * batch_x.size(0)
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == targets).sum().item()
            train_total += targets.size(0)
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch_x, batch_y in valid_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                
                outputs = model(batch_x)
                targets = torch.argmax(batch_y, dim=1)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item() * batch_x.size(0)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == targets).sum().item()
                val_total += targets.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        
        # Learning rate scheduling
        scheduler.step(val_acc)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
        
        # Record history
        history['loss'].append(train_loss)
        history['accuracy'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)
        
        # Print progress
        print(f"Epoch {epoch+1:3d}/{epochs} - "
              f"loss: {train_loss:.4f} - acc: {train_acc:.4f} - "
              f"val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}")
        
        # Early stopping check
        if early_stopping:
            early_stopping(val_acc)
            if early_stopping.early_stop:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break
    
    # Load best model
    model.load_state_dict(torch.load(save_path))
    print(f"\nBest validation accuracy: {best_val_acc:.4f}")
    
    return history

In [ ]:
def evaluate_model(model, data_x, data_y, batch_size=50):
    """Evaluate model on given data."""
    model.eval()
    model.to(device)
    
    data_x_t = torch.FloatTensor(data_x)
    data_y_t = torch.FloatTensor(data_y)
    dataset = TensorDataset(data_x_t, data_y_t)
    loader = TorchDataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    criterion = nn.CrossEntropyLoss()
    
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            outputs = model(batch_x)
            targets = torch.argmax(batch_y, dim=1)
            loss = criterion(outputs, targets)
            
            total_loss += loss.item() * batch_x.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
            
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    
    loss = total_loss / total
    accuracy = correct / total
    
    return loss, accuracy, np.array(all_preds), np.array(all_targets)


def matrices(model, data_x, data_y, name):
    """Evaluate and print metrics (matches original function)."""
    loss, acc, preds, targets = evaluate_model(model, data_x, data_y)
    print(f"{len(data_x)//50 + 1}/{len(data_x)//50 + 1} - loss: {loss:.4f} - accuracy: {acc:.4f}")
    print(f"{name}_best_acc: {acc}")
    return acc

In [ ]:
def plot_history(history):
    """Plot training history."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    ax1.plot(history['loss'], label='Train', linewidth=2)
    ax1.plot(history['val_loss'], label='Validation', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14)
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Accuracy
    ax2.plot(history['accuracy'], label='Train', linewidth=2)
    ax2.plot(history['val_accuracy'], label='Validation', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

---
## Training and Evaluation

The cells below demonstrate training on different datasets.

**Note:** You need to have the data files in the correct paths:
- SAT-6: `Data/SAT-6/sat-6-full.mat`
- LCZ: `Data/LCZ/data_5fold_5classes.mat`
- Overhead: `Data/overhead/training/` and `Data/overhead/testing/`
- CIFAR-10: `Data/cifar-10-batches-py/`

In [ ]:
# Set random seed for reproducibility
set_seed(42)

In [ ]:
# =============================================================================
# EXAMPLE: CIFAR-10 TRAINING
# =============================================================================

dataset = 'cifar10'

# Load data
print(f"Loading {dataset} dataset...")
dataloader = DataLoader(dataset)
train_x, train_y, valid_x, valid_y, test_x, test_y = dataloader.get_data()
class_name = dataloader.get_categories()

# For CIFAR-10, use descriptive class names
if dataset == 'cifar10':
    class_name = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Train: {train_x.shape}, {train_y.shape}")
print(f"Valid: {valid_x.shape}, {valid_y.shape}")
print(f"Test: {test_x.shape}, {test_y.shape}")
print(f"Classes: {class_name}")

# Visualize samples
vis_samples(train_x[:5], train_y[:5], class_name)

In [ ]:
# Build model
n_classes = train_y.shape[1]
n_channels = train_x.shape[-1]

seqnn_model = build_SEQNN_model(n_classes, n_channels, dataset)

In [ ]:
# Train model
# Use smaller subset for testing (remove limits for full training)
train_subset = 500   # Set to len(train_x) for full training
valid_subset = 100   # Set to len(valid_x) for full training

print(f"\nStarting training on {dataset}...")
print(f"Training samples: {train_subset}, Validation samples: {valid_subset}")
print("-" * 60)

history = train_model(
    seqnn_model,
    train_x[:train_subset], train_y[:train_subset],
    valid_x[:valid_subset], valid_y[:valid_subset],
    epochs=10,        # Use 200 for full training (paper setting)
    batch_size=50,    # Paper setting
    lr=0.01,          # Paper setting
    save_path=f'trained_models/{dataset}_pytorch_model.pth',
    use_early_stopping=True
)

print("\nTraining complete!")

In [ ]:
# Plot training history
plot_history(history)

In [ ]:
# Evaluate on all splits
test_subset = 100  # Set to len(test_x) for full evaluation

print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

matrices(seqnn_model, train_x[:train_subset], train_y[:train_subset], 'train')
print()
matrices(seqnn_model, valid_x[:valid_subset], valid_y[:valid_subset], 'val')
print()
matrices(seqnn_model, test_x[:test_subset], test_y[:test_subset], 'test')

In [ ]:
# Save final model
os.makedirs('trained_models', exist_ok=True)
torch.save(seqnn_model.state_dict(), f'trained_models/{dataset}_final_weights.pth')
print(f"Model saved to trained_models/{dataset}_final_weights.pth")

---
## Additional Dataset Examples

Uncomment and run these cells for other datasets.

In [ ]:
# # =============================================================================
# # OVERHEAD DATASET
# # =============================================================================
# 
# dataset = 'overhead'
# 
# dataloader = DataLoader(dataset)
# train_x, train_y, valid_x, valid_y, test_x, test_y = dataloader.get_data()
# class_name = dataloader.get_categories()
# 
# print(f"Train: {train_x.shape}, Valid: {valid_x.shape}, Test: {test_x.shape}")
# print(f"Classes: {class_name}")
# 
# vis_samples(train_x[:5], train_y[:5], class_name)
# 
# n_classes = train_y.shape[1]
# n_channels = train_x.shape[-1]
# 
# seqnn_model = build_SEQNN_model(n_classes, n_channels, dataset)
# 
# # Train
# history = train_model(
#     seqnn_model, train_x, train_y, valid_x, valid_y,
#     epochs=200, batch_size=50, lr=0.01,
#     save_path=f'trained_models/{dataset}_pytorch_model.pth'
# )
# 
# # Evaluate
# matrices(seqnn_model, train_x, train_y, 'train')
# matrices(seqnn_model, valid_x, valid_y, 'val')
# matrices(seqnn_model, test_x, test_y, 'test')

In [ ]:
# # =============================================================================
# # SAT-6 DATASET
# # =============================================================================
# 
# dataset = 'sat'
# 
# dataloader = DataLoader(dataset)
# train_x, train_y, valid_x, valid_y, test_x, test_y = dataloader.get_data()
# class_name = dataloader.get_categories()
# 
# print(f"Train: {train_x.shape}, Valid: {valid_x.shape}, Test: {test_x.shape}")
# print(f"Classes: {class_name}")
# 
# vis_samples(train_x[:5], train_y[:5], class_name)
# 
# n_classes = train_y.shape[1]
# n_channels = train_x.shape[-1]
# 
# seqnn_model = build_SEQNN_model(n_classes, n_channels, dataset)
# 
# # Train
# history = train_model(
#     seqnn_model, train_x, train_y, valid_x, valid_y,
#     epochs=200, batch_size=50, lr=0.01,
#     save_path=f'trained_models/{dataset}_pytorch_model.pth'
# )
# 
# # Evaluate
# matrices(seqnn_model, train_x, train_y, 'train')
# matrices(seqnn_model, valid_x, valid_y, 'val')
# matrices(seqnn_model, test_x, test_y, 'test')

---
## Loading Pre-trained Weights

In [ ]:
# Example: Load pre-trained weights
# 
# model_path = f'trained_models/{dataset}_pytorch_model.pth'
# seqnn_model.load_state_dict(torch.load(model_path))
# print(f"Loaded weights from {model_path}")

---
## Conversion Notes

### What's Preserved:
1. **DataLoader**: Exact implementation from original `seqnn_dataLoader.py`
2. **Architecture**: Superpixel → EncodingPQC → Dense (same layer structure)
3. **Hyperparameters**: nElements=9, nEncodings=1, nQconv=1, pool=4
4. **Training**: Adam optimizer, lr=0.01, batch_size=50, epochs=200

### What's Different:
1. **Quantum Framework**: Cirq + TFQ → PennyLane
2. **Quantum Circuit**: Simplified variational circuit (original uses complex 6-control gates)
3. **Measurements**: Pauli observables instead of projector observables

### Performance Tips:
- Use `lightning.qubit` for faster simulation: `qml.device('lightning.qubit', wires=n_qubits)`
- For GPU: `qml.device('lightning.gpu', wires=n_qubits)` (requires CUDA)
- Reduce batch size if memory is limited